# Task 2 - Support Vector Machine

Dans ce notebook, nous allons implémenter un modèle de machines à vecteurs de support (SVM), un algorithme puissant en machine learning supervisé pour la classification et la régression, basé sur la recherche d'hyperplans optimaux.

## 1. Importation des bibliothèques nécessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
%matplotlib inline

## 2. Chargement et exploration des données

Pour cette démonstration, nous utiliserons le jeu de données du cancer du sein de Wisconsin, qui est un problème de classification binaire classique.

In [ ]:
# Chargement des données du cancer du sein de Wisconsin
breast_cancer = load_breast_cancer()

# Création d'un DataFrame pandas
df = pd.DataFrame(data=breast_cancer.data, columns=breast_cancer.feature_names)
df['target'] = breast_cancer.target

df.head()

In [ ]:
# Exploration des données
print("Shape of dataset:", df.shape)
print("\nInfo of dataset:")
df.info()

In [ ]:
# Statistiques descriptives
df.describe()

In [ ]:
# Vérification de la distribution des classes (variable cible)
target_column = df.columns[-1]
print(f"Variable cible: {target_column}")
print("\nDistribution des classes:")
print(df[target_column].value_counts())

# Visualisation de la distribution des classes
plt.figure(figsize=(8, 6))
df[target_column].value_counts().plot(kind='bar')
plt.title('Distribution des classes')
plt.xlabel('Classes')
plt.ylabel('Nombre d\'échantillons')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

## 3. Prétraitement des données

Séparation des caractéristiques (features) et de la variable cible, puis normalisation des données (essentielle pour les SVM).

In [ ]:
# Séparation des caractéristiques (features) et de la variable cible
X = df.iloc[:, :-1]  # Toutes les colonnes sauf la dernière
y = df.iloc[:, -1]   # Dernière colonne (variable cible)

print(f"Dimensions des caractéristiques: {X.shape}")
print(f"Dimensions de la variable cible: {y.shape}")

In [ ]:
# Normalisation des données (très importante pour les SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertir en DataFrame pour faciliter la manipulation
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print("Données après normalisation:")
X_scaled.head()

## 4. Division des données

Division des données en ensembles d'entraînement et de test.

In [ ]:
# Division des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Taille de l'ensemble d'entraînement: {X_train.shape}")
print(f"Taille de l'ensemble de test: {X_test.shape}")

## 5. Entraînement du modèle SVM avec noyau linéaire

Création et entraînement du modèle SVM avec un noyau linéaire.

In [ ]:
# Création et entraînement du modèle SVM avec noyau linéaire
svm_linear = SVC(kernel='linear', random_state=42, probability=True)
svm_linear.fit(X_train, y_train)

## 6. Prédiction et évaluation du modèle SVM linéaire

Prédiction sur l'ensemble de test et évaluation des performances du modèle.

In [ ]:
# Prédiction sur l'ensemble de test
y_pred_linear = svm_linear.predict(X_test)
y_pred_proba_linear = svm_linear.predict_proba(X_test)[:, 1]  # Probabilités pour la classe positive

# Affichage des premières prédictions
results_df = pd.DataFrame({'Valeurs réelles': y_test, 'Prédictions': y_pred_linear, 'Probabilités': y_pred_proba_linear})
print("Comparaison des valeurs réelles et prédites (SVM linéaire):")
print(results_df.head(10))

In [ ]:
# Évaluation du modèle
accuracy_linear = accuracy_score(y_test, y_pred_linear)
print(f"Précision du modèle SVM linéaire: {accuracy_linear:.4f}")

# Rapport de classification détaillé
print("\nRapport de classification (SVM linéaire):")
print(classification_report(y_test, y_pred_linear))

In [ ]:
# Matrice de confusion
cm_linear = confusion_matrix(y_test, y_pred_linear)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_linear, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion (SVM linéaire)')
plt.xlabel('Prédictions')
plt.ylabel('Valeurs réelles')
plt.show()

In [ ]:
# Courbe ROC et AUC
fpr_linear, tpr_linear, thresholds_linear = roc_curve(y_test, y_pred_proba_linear)
roc_auc_linear = auc(fpr_linear, tpr_linear)

plt.figure(figsize=(8, 6))
plt.plot(fpr_linear, tpr_linear, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc_linear:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Classifieur aléatoire')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC (SVM linéaire)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Aire sous la courbe ROC (AUC) - SVM linéaire: {roc_auc_linear:.4f}")

## 7. Entraînement du modèle SVM avec noyau RBF

Création et entraînement du modèle SVM avec un noyau RBF (radial basis function).

In [ ]:
# Création et entraînement du modèle SVM avec noyau RBF
svm_rbf = SVC(kernel='rbf', random_state=42, probability=True)
svm_rbf.fit(X_train, y_train)

## 8. Prédiction et évaluation du modèle SVM RBF

Prédiction sur l'ensemble de test et évaluation des performances du modèle.

In [ ]:
# Prédiction sur l'ensemble de test
y_pred_rbf = svm_rbf.predict(X_test)
y_pred_proba_rbf = svm_rbf.predict_proba(X_test)[:, 1]  # Probabilités pour la classe positive

# Affichage des premières prédictions
results_df = pd.DataFrame({'Valeurs réelles': y_test, 'Prédictions': y_pred_rbf, 'Probabilités': y_pred_proba_rbf})
print("Comparaison des valeurs réelles et prédites (SVM RBF):")
print(results_df.head(10))

In [ ]:
# Évaluation du modèle
accuracy_rbf = accuracy_score(y_test, y_pred_rbf)
print(f"Précision du modèle SVM RBF: {accuracy_rbf:.4f}")

# Rapport de classification détaillé
print("\nRapport de classification (SVM RBF):")
print(classification_report(y_test, y_pred_rbf))

In [ ]:
# Matrice de confusion
cm_rbf = confusion_matrix(y_test, y_pred_rbf)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rbf, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion (SVM RBF)')
plt.xlabel('Prédictions')
plt.ylabel('Valeurs réelles')
plt.show()

In [ ]:
# Courbe ROC et AUC
fpr_rbf, tpr_rbf, thresholds_rbf = roc_curve(y_test, y_pred_proba_rbf)
roc_auc_rbf = auc(fpr_rbf, tpr_rbf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rbf, tpr_rbf, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc_rbf:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Classifieur aléatoire')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC (SVM RBF)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Aire sous la courbe ROC (AUC) - SVM RBF: {roc_auc_rbf:.4f}")

## 9. Optimisation des hyperparamètres pour SVM RBF

Utilisation de GridSearchCV pour trouver les meilleurs hyperparamètres pour le noyau RBF.

In [ ]:
# Définition de la grille de paramètres
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

# Recherche des meilleurs paramètres
grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf', random_state=42, probability=True),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Meilleurs paramètres trouvés pour SVM RBF:")
print(grid_search.best_params_)
print(f"\nMeilleur score de validation croisée: {grid_search.best_score_:.4f}")

In [ ]:
# Entraînement du modèle avec les meilleurs paramètres
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test)
y_pred_proba_best = best_svm.predict_proba(X_test)[:, 1]

# Évaluation du modèle optimisé
accuracy_best = accuracy_score(y_test, y_pred_best)
print(f"Précision du modèle SVM RBF optimisé: {accuracy_best:.4f}")

print("\nRapport de classification du modèle SVM RBF optimisé:")
print(classification_report(y_test, y_pred_best))

In [ ]:
# Courbe ROC et AUC du modèle optimisé
fpr_best, tpr_best, thresholds_best = roc_curve(y_test, y_pred_proba_best)
roc_auc_best = auc(fpr_best, tpr_best)

plt.figure(figsize=(8, 6))
plt.plot(fpr_best, tpr_best, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc_best:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Classifieur aléatoire')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC (SVM RBF optimisé)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Aire sous la courbe ROC (AUC) - SVM RBF optimisé: {roc_auc_best:.4f}")

## 10. Comparaison des différents noyaux

Comparaison des performances entre différents noyaux SVM.

In [ ]:
# Test de différents noyaux
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
svm_models = {}
accuracies = {}
roc_aucs = {}

for kernel in kernels:
    # Pour le noyau polynomial, utilisons un degré faible pour éviter le sur-apprentissage
    if kernel == 'poly':
        svm = SVC(kernel=kernel, degree=2, random_state=42, probability=True)
    else:
        svm = SVC(kernel=kernel, random_state=42, probability=True)
    
    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)
    y_pred_proba = svm.predict_proba(X_test)[:, 1]
    
    svm_models[kernel] = svm
    accuracies[kernel] = accuracy_score(y_test, y_pred)
    
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_aucs[kernel] = auc(fpr, tpr)

# Affichage des résultats
print("Comparaison des noyaux SVM:")
print("-" * 40)
for kernel in kernels:
    print(f"{kernel.capitalize():10} | Précision: {accuracies[kernel]:.4f} | AUC: {roc_aucs[kernel]:.4f}")

In [ ]:
# Visualisation de la comparaison
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# Précision
ax[0].bar(accuracies.keys(), accuracies.values(), color=['skyblue', 'lightgreen', 'salmon', 'gold'])
ax[0].set_title('Précision par noyau SVM')
ax[0].set_ylabel('Précision')
ax[0].set_ylim([0.9, 1.0])
ax[0].grid(True, alpha=0.3)

# AUC
ax[1].bar(roc_aucs.keys(), roc_aucs.values(), color=['skyblue', 'lightgreen', 'salmon', 'gold'])
ax[1].set_title('AUC par noyau SVM')
ax[1].set_ylabel('AUC')
ax[1].set_ylim([0.9, 1.0])
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

Dans ce notebook, nous avons implémenté des modèles de machines à vecteurs de support :
- Chargement et exploration des données du cancer du sein
- Prétraitement des données avec normalisation (essentielle pour les SVM)
- Division des données en ensembles d'entraînement et de test
- Entraînement de modèles SVM avec différents noyaux (linéaire, RBF)
- Évaluation des modèles avec plusieurs métriques (précision, rappel, F1-score, AUC)
- Visualisation des résultats (matrices de confusion, courbes ROC)
- Optimisation des hyperparamètres avec GridSearchCV
- Comparaison des performances entre différents noyaux

Les SVM sont des algorithmes puissants qui fonctionnent particulièrement bien dans des espaces de grande dimension. La normalisation des données est cruciale pour leur bon fonctionnement. Le choix du noyau et le réglage des hyperparamètres (C, gamma) ont un impact majeur sur les performances du modèle.